## Structured Output
- Models can be requested to provide the responses in a format matching the given schema.
- useful for parsing the output that can be used in subsequent prsing.
- Techniques used
  - Pydantic
  - set with field validation, desc, nestedStructures

In [8]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001888A2B5950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001888A2B6350>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [25]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(..., description="title of the movie")
    year: int = Field(..., description="release year of the movie")
    director: str = Field(..., description="director of the movie")
    rating: float = Field(..., description="rating of the movie")
    desc: str = Field(..., description="what the movie is about")

model_with_structure = model.with_structured_output(Movie)
# model_with_structure
# model.invoke("What is the title, release year, director, and rating of the movie RRR?")
response = model_with_structure.invoke("What is the title, release year, director, and rating of the movie RRR?")
print(response)




title='RRR' year=2022 director='S.S. Rajamouli' rating=8.8 desc='A historical action film about two revolutionaries from different backgrounds who join forces to fight against British colonial rule in India.'


### Msg output alongside the parsed structure

In [ ]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(..., description="title of the movie")
    year: int = Field(..., description="release year of the movie")
    director: str = Field(..., description="director of the movie")
    rating: float = Field(..., description="rating of the movie")
    desc: str = Field(..., description="what the movie is about")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("What is the title, release year, director, and rating of the movie RRR?")
print(response)

In [29]:
### Nested Structure
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name: str = Field(..., description="name of the actor")
    role: str = Field(..., description="role of the actor in the movie")

class Movie(BaseModel):
    title: str = Field(..., description="title of the movie")
    year: int = Field(..., description="release year of the movie")
    director: str = Field(..., description="director of the movie")
    rating: float = Field(..., description="rating of the movie")
    desc: str = Field(..., description="what the movie is about")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details for the movie RRR?")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie RRR. Let me see. I need to use the Movie function provided. The required parameters are title, year, director, rating, and desc. First, I should confirm the title is "RRR". The release year was 2022. The directors are S. S. Rajamouli. The rating is probably high, maybe around 8.5 or so. The description should mention it\'s an action film with a unique story involving two friends. I need to make sure all required fields are filled. Let me check again: title, year, director, rating, desc. Yes, that\'s all covered. I\'ll structure the JSON accordingly.\n', 'tool_calls': [{'id': 'th554n16h', 'function': {'arguments': '{"desc":"RRR is a 2022 Indian Telugu-language action film written and directed by S. S. Rajamouli. It stars Jr. NTR and Ram Charan in lead roles, portraying a fictional friendship between two revolutionaries in the 1920s. The film is known for its intens

In [12]:
### Typed Dict
from typing_extensions import TypedDict,Annotated
class MovieDict(TypedDict):
    """A Movie with details"""
    title: Annotated[str,...,"title of the movie"]
    year: Annotated[int, ...,"release year of the movie"]
    director: Annotated[str, ...,"director of the movie"]
    rating: Annotated[float, ...,"rating of the movie"]
    desc: Annotated[str, ...,"what the movie is about"]
model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details for the movie RRR?")
print(response)

{'desc': 'A historical action film depicting two Indian revolutionaries from different regions forming an alliance against the British Empire in the 1920s, featuring extravagant dance sequences and dramatic storytelling.', 'director': 'S. S. Rajamouli', 'rating': 8.8, 'title': 'RRR', 'year': 2022}


In [25]:
### DataClasses

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class contactInfo(BaseModel):
    name: str = Field(..., description="name of the contact")
    email: str = Field(..., description="email of the contact")
    phone: str = Field(..., description="phone number of the contact")

agent = create_agent(
    model, 
    response_format=contactInfo
)
response = agent.invoke({"messages": [{"role": "user", "content": "Extract contact information from the following text: 'John Doe, email: john.doe@example.com, phone: 123-456-7890'"}]})
print(response["structured_response"])
# print(response["HumanMessage"])
print(response)



name='John Doe' email='john.doe@example.com' phone='123-456-7890'
{'messages': [HumanMessage(content="Extract contact information from the following text: 'John Doe, email: john.doe@example.com, phone: 123-456-7890'", additional_kwargs={}, response_metadata={}, id='321e864d-400a-41de-9345-041729718151'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's see. The user wants me to extract contact information from the given text. The text is 'John Doe, email: john.doe@example.com, phone: 123-456-7890'. I need to use the contactInfo function they provided.\n\nFirst, I'll break down the text. The name is John Doe. Then the email is after 'email: ', so that's john.doe@example.com. The phone number is after 'phone: ', which is 123-456-7890. \n\nI should check the required parameters for the contactInfo function. The required fields are name, email, and phone. All three are present here. \n\nI need to make sure the email is correctly formatted. It has an '@' and a doma

In [27]:
from dataclasses import dataclass
from langchain.agents import create_agent
@dataclass
class contactInfo:
    """A contact with details"""
    name: str
    email: str
    phone: str
agent = create_agent(
    model,
    response_format=contactInfo
)

response = agent.invoke({"messages": [{"role": "user", "content": "Extract contact information from the following text: 'John Doe, email: john.doe@example.com, phone: 123-456-7890'"}]})
print(response['structured_response'])

contactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')
